<p>
<h1><b><center></center></b></h1>
<center><img src="https://drive.google.com/uc?id=1UJc1ci41G6ahJ7ProKvunUOIBcTXZ6ZG" align="center" width="550"></center>
</p>
<h1><b><center>Mecánica Celeste</center></b></h1>
<h2><b><center>Prof. Jorge I. Zuluaga</center></b></h1>
<h2><b><center>Proyecto del Curso</center></b><h2>
<h3><b><center>Disco de acreción alrededor de un agujero negro</center></b><h3>
<h7><center><i>Última actualización del profesor</b>: Viernes 20 de marzo de 2026, 2:00 pm</i></center><h7>
</p>

<center>
<img src="https://cdn.sci.news/images/2021/02/image_9386-AT2019dsg.jpg" align="center" width="100%"></center>

Muchos objetos astrofísicos fascinantes (núcleos galácticos activos, hipernovas, binarias de rayos X) pueden explicarse con los fenómenos que ocurren en los denominados discos de acreción alrededor de agujeros negros. En este proyecto usaremos lo visto en el curso de Relatividad y Gravitación para estudiar esos fenómenos.

El objetivo de este trabajo es desarrollar simulaciones, cálculos útiles, predicciones sobre el comportamiento de las partículas en un disco de acreción. La idea es poner en práctica lo visto en el curso, tanto en las partes de relatividad especial como en las de relatividad general, para estudiar estos sistemas.

## Algunas ideas

Existen muchas maneras de aplicar la teoría que veremos en el curso en este problema y no queremos sesgar tu elección de los temas o cálculos que quieras escoger para aplicarla. Sin embargo aquí van algunas ideas de cálculos que se podrían hacer:
- Simulación del movimiento de muchas partículas alrededor del agujero negro usando, inicialmente, la dinámica newtoniana.
- Cálculo y representación gráfica de la luz emitida por el disco y observada desde distinto ángulos, incluyendo los efectos de beaming y desplazamiento espectral.
- Simulación del jet del agujero negro usando partículas cargadas moviéndose en el campo del jet.
- Cálculo de la trayectoria de fotones para visualizar el disco de acreción desde distintos ángulos.
- Cálculo de las trayectorias de partículas usando la métrica de Schwarzschild y de Kerr.

## Entregables

El entregable del proyecto es **un notebook final de Jupyter** con una descripción de la teoría básica que uses, los experimentos numéricos que hayas realizado y las conclusiones a las que llegues con esos experimentos. Por supuesto puedes desarrollar otros programas y notebooks paralelos, pero se revisará el notebook con el reporte final.

Adicionalmente se deberá entregar **un repositorio de GitHub** que tenga todos los archivos, datos, notebooks, programas usados para este propósito. El notebook debe estar alojado en el repositorio.

## Desarrollo del proyecto: primera idea con REBOUND

En este documento desarrollaré el proyecto final de Relatividad y Gravitación para la carrera de Astronomía.

### Idea inicial
Simulación del movimiento de muchas partículas alrededor de un agujero negro usando, inicialmente, dinámica newtoniana.

### Objetivo de esta primera versión
1. Modelar el agujero negro como una masa puntual central.
2. Inicializar muchas partículas de prueba en un disco.
3. Integrar sus órbitas con `REBOUND` y visualizar su evolución.

> Esta etapa es un punto de partida clásico para luego extender el modelo con efectos relativistas (por ejemplo, potencial pseudo-newtoniano o ecuaciones geodésicas).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rebound


def crear_simulacion_disco_newtoniano(
    n_particulas=400,
    masa_bh=1.0,
    r_min=1.0,
    r_max=6.0,
    dt=0.01,
    semilla=42,
):
    """Crea una simulacion newtoniana de un disco de particulas de prueba."""
    sim = rebound.Simulation()
    sim.G = 1.0
    sim.integrator = "whfast"
    sim.dt = dt

    # Agujero negro central (masa puntual)
    sim.add(m=masa_bh, hash="BH")

    rng = np.random.default_rng(semilla)

    for _ in range(n_particulas):
        r = rng.uniform(r_min, r_max)
        theta = rng.uniform(0.0, 2.0 * np.pi)

        x = r * np.cos(theta)
        y = r * np.sin(theta)

        # Velocidad circular newtoniana alrededor de una masa central
        v_circ = np.sqrt(sim.G * masa_bh / r)

        # Pequena dispersion para evitar un disco perfectamente frio
        factor = 1.0 + rng.normal(0.0, 0.03)
        v = max(0.0, v_circ * factor)

        vx = -v * np.sin(theta)
        vy = v * np.cos(theta)

        sim.add(m=0.0, x=x, y=y, z=0.0, vx=vx, vy=vy, vz=0.0)

    sim.move_to_com()
    return sim


sim = crear_simulacion_disco_newtoniano()

In [ ]:
# Integracion temporal y almacenamiento de trayectorias
n = sim.N - 1  # Excluye el agujero negro central

# En unidades adimensionales del problema
tiempo_total = 120.0
n_pasos = 1000
tiempos = np.linspace(0.0, tiempo_total, n_pasos)

trayectorias = np.zeros((n_pasos, n, 2))

for k, t in enumerate(tiempos):
    sim.integrate(t, exact_finish_time=0)
    for j in range(n):
        p = sim.particles[j + 1]
        trayectorias[k, j, 0] = p.x
        trayectorias[k, j, 1] = p.y

print(f"Simulacion completada: {n} particulas, {n_pasos} pasos")

In [ ]:
# Visualizacion de una muestra de orbitas y el estado final
fig, ax = plt.subplots(figsize=(8, 8))

n_muestra = min(60, n)
indices = np.linspace(0, n - 1, n_muestra, dtype=int)

for idx in indices:
    ax.plot(
        trayectorias[:, idx, 0],
        trayectorias[:, idx, 1],
        lw=0.8,
        alpha=0.6,
    )

# Agujero negro en el origen
ax.scatter(0.0, 0.0, s=120, c="black", label="Agujero negro")

# Posicion final de todas las particulas
ax.scatter(
    trayectorias[-1, :, 0],
    trayectorias[-1, :, 1],
    s=6,
    alpha=0.5,
    label="Particulas (estado final)",
)

ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Disco de particulas en potencial newtoniano")
ax.legend(loc="upper right")
ax.grid(alpha=0.2)
plt.show()

### Siguientes extensiones naturales
- Explorar estabilidad variando el radio inicial y la dispersión de velocidades.
- Incluir potencial pseudo-newtoniano de Paczynski-Wiita para aproximar efectos relativistas.
- Comparar órbitas newtonianas con órbitas calculadas desde métricas de Schwarzschild/Kerr.
- Estimar observables: distribución radial, tiempos característicos y estructura del disco.